# Analyze Solar PV Capacity Density of Existing Plants

### Overview

This notebook calculates the rated capacity density of existing solar PV facilities in the Western US.

### Data Requirements

The US Large-Scale Solar Photovoltaic Database shapefile must be downloaded prior to running this notebook as we do not provide the source data directly. The geospatial shapefile can be downloaded here: https://doi.org/10.5066/P9IA3TUS

Please extract the downloaded data inside the `data/input_data` directory of this repository as the paths in this notebook are set to that expectation.

* **Source Data Title:** United States Large-Scale Solar Photovoltaic Database
* **Description from Source:** The United States Large-Scale Solar Photovoltaic Database (USPVDB) provides the locations and array boundaries of U.S. ground-mounted photovoltaic (PV) facilities with capacity of 1 megawatt or more. It includes corresponding PV facility information, including panel type, site type, and initial year of operation.
* **Source URL:** https://doi.org/10.5066/P9IA3TUS
* **Date Accessed:** 02/01/25
* **Citation:** Fujita, K.S., Ancona, Z.H., Kramer, L.A., Straka, M., Gautreau, T.E., Garrity, C.P., Robson, D., Diffendorfer, J.E., and Hoen, B., 2023, United States Large-Scale Solar Photovoltaic Database (v2.0, August, 2024): U.S. Geological Survey and Lawrence Berkeley National Laboratory data release, https://doi.org/10.5066/P9IA3TUS.

## Imports

In [3]:
import geopandas as gpd
import pandas as pd
import plotly.express as px

## Data Paths

In [ ]:
# data dir
data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data', 'input_data')

# solar database file
solar_fn = os.path.join(data_dir, 'uspvdbSHP', 'uspvdb_v2_0_20240801.shp')

## Analysis

In [12]:
# direct land use assumption
direct_land_frac = .75

# read in solar shapefile
solar = gpd.read_file('/Users/mong275/Downloads/uspvdbSHP/uspvdb_v2_0_20240801.shp')

# convert to albers CRS
solar.to_crs('ESRI:102003', inplace=True)

# remove smaller solar plants to align with CERF assumed rated capacity range
solar = solar[solar.p_cap_dc > 10]

# remove non single-axis facilities
solar = solar[solar.p_axis == 'single-axis']

# select facilities operational between 2010 and 2020
solar = solar[solar.p_year > 2010]
#solar = solar[solar.p_year < 2020]

# select facilities in western US states
solar = solar[solar.p_state.isin(['CA', 'OR', 'WA', 'ID', 'NV', 'UT', 'MT', 'WY', 'AZ', 'CO', 'NM'])]

# determine the project area by assuming panel area in meters squared covers 75% of the total project area
solar['project_area_m2'] = solar['p_area'] * (1 / direct_land_frac)

# calculate the capacity density (MW/square meter) as the ratio of DC capacity and project area (m2)
solar['land_use_mw_m2'] = solar['p_cap_dc'] / solar['project_area_m2']

# calculate the capacity density per km2 (MW/km2) 
solar['land_use_mw_km2'] = (solar['land_use_mw_m2'] * 1e6)

fig = px.histogram(solar['land_use_mw_km2'], nbins=100)

fig.show()